In [10]:
!pip install -q -U pandas google-generativeai

import pandas as pd
import google.generativeai as genai
import time

genai.configure(api_key="INSERT_API_HERE")

model = genai.GenerativeModel("gemini-2.5-pro")

EXCEL_FILE_PATH = "/path/to/activities"
df = pd.read_excel(EXCEL_FILE_PATH)

results = []

for idx, row in df.iterrows():
    variables = str(row["Variables"])
    relationships = str(row["Relationships"])
    questions = str(row["Questions"])
    levels = [str(row[f"Level {i}"]) for i in range(1, 5)]

    prompt = f"""
You are an expert evaluator trained in STEM education, instructional design, and AI-assisted learning material generation. Your task is to score activities across five evaluation criteria.

You will receive input from an Excel file with the following columns:
- Variables: Key variables for the simulation
- Relationships: Relationships among the variables
- Questions: Simulation-related assessment questions
- Level 1 to Level 4: Sets of activities generated for increasing levels of prompt design

Each row represents one simulation. For each row, review the activities from Level 1 through Level 4 as separate sets.
Assign a score out of 10 **along with a justification of 1-2 sentences** for Level 1 through Level 4 (columns) for each of the five evaluation criteria below:

1. **Concept & Variable Coverage**: How well do the activities reflect and incorporate the listed variables and relationships?
2. **Semantic Relevance to Questions**: How well do the activities help students answer the provided questions?
3. **Conceptual Progression**: Does the activity set have a clear logical order where each step builds on the previous?
4. **Clarity & Design Precision**: Do the activities give clear instructions (e.g., “Vary X while keeping Y constant”, “measure Z and record”) and avoid vague phrases?
5. **Output Format Accuracy**: Are the activities clearly separated, numbered or bulleted?

Please respond with structured scores and brief explanations per level.

---
**Variables**:
{variables}

**Relationships**:
{relationships}

**Questions**:
{questions}

---
**Activity Sets**:

**Level 1**:
{levels[0]}

**Level 2**:
{levels[1]}

**Level 3**:
{levels[2]}

**Level 4**:
{levels[3]}
"""

    try:
        response = model.generate_content(prompt)
        results.append({
            "Row": idx,
            "Evaluation": response.text
        })
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        results.append({
            "Row": idx,
            "Evaluation": f"Error: {str(e)}"
        })

    time.sleep(2)

eval_df = pd.DataFrame(results)
eval_df.to_csv("llm_gemini_evaluation_results.csv", index=False)
print("Evaluation complete. Results saved to llm_gemini_evaluation_results.csv")

Evaluation complete. Results saved to llm_gemini_evaluation_results.csv
